# GTZAN Initial EDA

This notebook visualizes the local GTZAN dataset that lives at `data/genres_original`.

It covers:
- genre class distribution
- basic audio metadata checks
- one MFCC example per genre

## Environment Setup

Run this first. It finds the project root and installs missing requirements into the active notebook kernel if needed.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import importlib.util
import os
import subprocess
import sys


def find_repo_root(start: Path) -> Path:
    """Find the audio-genre-classifier repo from common notebook cwd values."""
    start = start.resolve()
    candidates = []
    for path in (start, *start.parents):
        candidates.append(path)
        candidates.append(path / "audio-genre-classifier")

    for candidate in candidates:
        if (candidate / "requirements.txt").exists() and (candidate / "code").exists():
            return candidate

    raise FileNotFoundError("Could not find the audio-genre-classifier repo root.")


repo_root = find_repo_root(Path.cwd())
os.chdir(repo_root)

os.environ.setdefault("MPLCONFIGDIR", str(repo_root / ".matplotlib-cache"))
os.environ.setdefault("NUMBA_CACHE_DIR", str(repo_root / ".numba-cache"))

required_modules = ["matplotlib", "pandas", "librosa"]
missing_modules = [
    module for module in required_modules if importlib.util.find_spec(module) is None
]

if missing_modules:
    print(f"Installing missing modules into this kernel: {missing_modules}")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-r", str(repo_root / "requirements.txt")]
    )
else:
    print("Notebook kernel already has the required modules.")

dataset_path = repo_root / "data" / "genres_original"
report_dir = repo_root / "reports"
report_dir.mkdir(exist_ok=True)
print(f"Project root: {repo_root}")
print(f"Dataset path: {dataset_path}")

## Notebook Helper Functions

These helpers keep the EDA cells below compact while keeping the notebook self-contained.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


EXPECTED_GENRES = {
    "blues",
    "classical",
    "country",
    "disco",
    "hiphop",
    "jazz",
    "metal",
    "pop",
    "reggae",
    "rock",
}


def discover_audio_files(dataset_path: Path) -> pd.DataFrame:
    """Return one row per GTZAN .wav file with its genre label."""
    if not dataset_path.exists():
        raise FileNotFoundError(
            f"Dataset path not found: {dataset_path}. Place GTZAN at "
            "data/genres_original before running this notebook."
        )

    rows = []
    for genre_dir in sorted(path for path in dataset_path.iterdir() if path.is_dir()):
        for wav_file in sorted(genre_dir.glob("*.wav")):
            rows.append(
                {
                    "genre": genre_dir.name,
                    "file_name": wav_file.name,
                    "file_path": str(wav_file),
                }
            )

    return pd.DataFrame(rows, columns=["genre", "file_name", "file_path"])


def build_genre_counts(dataset_path: Path) -> pd.DataFrame:
    """Count .wav files in each genre folder and return a tidy DataFrame."""
    audio_files = discover_audio_files(dataset_path)
    return (
        audio_files.groupby("genre", as_index=False)
        .size()
        .rename(columns={"size": "count"})
        .sort_values("genre")
        .reset_index(drop=True)
    )


def validate_gtzan_layout(genre_counts: pd.DataFrame) -> pd.DataFrame:
    """Return expected genre counts with a status column for quick QA."""
    counts_by_genre = dict(zip(genre_counts["genre"], genre_counts["count"]))
    rows = []
    for genre in sorted(EXPECTED_GENRES | set(counts_by_genre)):
        count = counts_by_genre.get(genre, 0)
        if genre not in EXPECTED_GENRES:
            status = "unexpected genre"
        elif count == 100:
            status = "ok"
        elif count == 0:
            status = "missing genre"
        else:
            status = "expected 100 files"

        rows.append({"genre": genre, "count": count, "status": status})

    return pd.DataFrame(rows, columns=["genre", "count", "status"])


def plot_genre_distribution(
    genre_counts: pd.DataFrame,
    output_path: Path,
) -> None:
    """Save a genre class distribution bar chart."""
    ax = genre_counts.plot(
        kind="bar",
        x="genre",
        y="count",
        legend=False,
        color="steelblue",
        figsize=(10, 5),
    )
    ax.set_title("GTZAN Genre Class Distribution")
    ax.set_xlabel("Genre")
    ax.set_ylabel("Number of .wav files")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=150)
    plt.close("all")


def build_audio_metadata(
    dataset_path: Path,
    limit: int | None = None,
) -> pd.DataFrame:
    """Load GTZAN files with librosa and return basic audio metadata."""
    import librosa

    audio_files = discover_audio_files(dataset_path)
    if limit is not None:
        audio_files = audio_files.head(limit)

    rows = []
    for row in audio_files.itertuples(index=False):
        samples, sample_rate = librosa.load(row.file_path, sr=None, mono=True)
        rows.append(
            {
                "genre": row.genre,
                "file_name": row.file_name,
                "sample_rate": sample_rate,
                "sample_count": len(samples),
                "duration_seconds": librosa.get_duration(
                    y=samples,
                    sr=sample_rate,
                ),
            }
        )

    return pd.DataFrame(rows)


def build_example_file_paths(dataset_path: Path) -> dict[str, Path]:
    """Choose one deterministic example .wav file per genre."""
    audio_files = discover_audio_files(dataset_path)
    examples = {}

    for genre in sorted(EXPECTED_GENRES):
        genre_files = audio_files[audio_files["genre"] == genre].sort_values("file_name")
        if not genre_files.empty:
            examples[genre] = Path(genre_files.iloc[0]["file_path"])

    return examples


def plot_mfcc_examples(
    dataset_path: Path,
    output_path: Path,
    n_mfcc: int = 13,
    duration: float = 30.0,
) -> None:
    """Load one audio example per genre and save MFCC plots."""
    import librosa
    import librosa.display

    examples = build_example_file_paths(dataset_path)
    fig, axes = plt.subplots(5, 2, figsize=(12, 14), constrained_layout=True)
    axes = axes.flatten()

    image = None
    for ax, genre in zip(axes, sorted(examples)):
        samples, sample_rate = librosa.load(
            examples[genre],
            sr=None,
            mono=True,
            duration=duration,
        )
        mfccs = librosa.feature.mfcc(
            y=samples,
            sr=sample_rate,
            n_mfcc=n_mfcc,
        )
        image = librosa.display.specshow(
            mfccs,
            x_axis="time",
            ax=ax,
            cmap="magma",
        )
        ax.set_title(f"{genre}: {examples[genre].name}")
        ax.set_ylabel("MFCC")
        ax.set_xlabel("Time")

    for ax in axes[len(examples) :]:
        ax.axis("off")

    fig.suptitle("GTZAN MFCC Examples by Genre", fontsize=16)
    if image is not None:
        fig.colorbar(image, ax=axes, format="%+2.0f dB", shrink=0.65)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=150)
    plt.close(fig)

## Genre Class Distribution

GTZAN should have 10 genre folders with 100 `.wav` files per genre.

In [ ]:
genre_counts = build_genre_counts(dataset_path)
display(genre_counts)

In [ ]:
layout_validation = validate_gtzan_layout(genre_counts)
display(layout_validation)

In [ ]:
genre_chart_path = report_dir / "notebook_genre_distribution.png"
plot_genre_distribution(genre_counts, output_path=genre_chart_path)
display(Image(filename=str(genre_chart_path)))

## Audio Metadata Smoke Check

This loads a small sample with `librosa` so the notebook stays quick while still confirming audio files can be read.

In [ ]:
metadata_sample = build_audio_metadata(dataset_path, limit=20)
display(metadata_sample.head())

In [ ]:
metadata_summary = (
    metadata_sample.groupby("genre")
    .agg(
        files=("file_name", "count"),
        avg_duration_seconds=("duration_seconds", "mean"),
        min_sample_rate=("sample_rate", "min"),
        max_sample_rate=("sample_rate", "max"),
    )
    .round(2)
)
display(metadata_summary)

## MFCC Examples By Genre

This renders one deterministic example file per genre using the first `.wav` file in each folder.

In [ ]:
mfcc_chart_path = report_dir / "notebook_mfcc_examples_by_genre.png"
plot_mfcc_examples(dataset_path=dataset_path, output_path=mfcc_chart_path)
display(Image(filename=str(mfcc_chart_path)))